# Long-Term Environmental and Biological Monitoring Measurements from Basque Country Estuaries and Coasts, 1995–2014 Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qky9-b9vy/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

# Print basic metadata
print(f"Dataset name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Keywords: {', '.join(metadata['keywords'])}")
print(f"Temporal Coverage: {metadata['temporalCoverage']}")
print(f"Spatial Coverage: {metadata['spatialCoverage']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their @id
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs.get('name', '')}")

# Explore fields for the first record set
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    fields = record_sets[0].get('field', [])
    print(f"Fields in record set {example_record_set_id}:")
    for field in fields:
        print(f"  - {field['@id']}: {field.get('name', '')} (Type: {field.get('dataType', '')})")
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If you have multiple record sets, list their @id's here
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Extract records from each record set (@id)
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Preview columns and first rows for the first record set
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"Columns for record set {first_record_set_id}: {dataframes[first_record_set_id].columns.tolist()}")
    display(dataframes[first_record_set_id].head())
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Find a numeric field for filtering and analysis
df = dataframes[first_record_set_id]

numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
if numeric_columns:
    numeric_field_id = numeric_columns[0]  # Use the first numeric column
    print(f"Using numeric field '{numeric_field_id}' for filtering.")

    # Set an example threshold
    threshold = 10  # Example threshold; adjust based on your data
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field, add new column
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    # Try grouping by a categorical field
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if categorical_columns:
        group_field_id = categorical_columns[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped average of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No categorical columns found for grouping.")
else:
    print("No numeric columns available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for normalized numeric field
if numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=30, color='dodgerblue')
    plt.title(f'Normalized Distribution of {numeric_field_id}')
    plt.xlabel(f'{numeric_field_id}_normalized')
    plt.ylabel('Frequency')
    plt.show()

# If grouped_df exists, plot grouped mean
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id], palette='viridis')
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The Basque Country long-term environmental dataset provides rich measurements across several stations and years, accessible through the Croissant schema.
- Using `mlcroissant`, we loaded metadata, identified record sets, extracted tabular records and explored numeric and categorical data fields.
- Basic filtering, normalization, and grouping illustrated common EDA strategies.
- Visualizations reveal patterns in the normalized data and highlight possible station or category-level trends for further investigation.
- This notebook can be customized to explore specific variables and record sets using their unique `@id` references.
